In [ ]:
import glob
import json
import os
import random
import uuid
from itertools import product

import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd
from scipy.signal import stft, welch
from scipy.stats import entropy, norm

# Scikit-Learn
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from sklearn.model_selection import KFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight

# TensorFlow / Keras & PyTorch
import tensorflow as tf
from tensorflow.keras import Model, callbacks, layers, models
import torch

# ---------------------------------------------------------------------------
# REPRODUCIBILITY & HARDWARE SETUP
# ---------------------------------------------------------------------------
print("Script execution started...")
SEED = 42

os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['TF_DETERMINISTIC_OPS'] = '1'
os.environ['TF_CUDNN_DETERMINISTIC'] = '1'

# Set seeds for Python, NumPy, TensorFlow, and PyTorch
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
tf.config.threading.set_inter_op_parallelism_threads(1)
tf.config.threading.set_intra_op_parallelism_threads(1)

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

print(f"Reproducibility settings locked with SEED: {SEED}")

# ---------------------------------------------------------------------------
# HARDWARE ACCELERATION CHECK
# ---------------------------------------------------------------------------
if tf.config.list_physical_devices('GPU'):
    print("TensorFlow GPU Accelerated Backend Active.")
else:
    print("No GPU detected for TensorFlow. Using CPU.")

if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"CUDA GPU Accelerated Backend Active: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device("cpu")

In [ ]:
# Path to the participants TSV file
file_path = "/kaggle/input/datasets/adithyarajnarayanan/eeg-pd-dataset-part-3/EEG dataset part 3 /participants.tsv"

# Read the tab-separated file
df = pd.read_csv(file_path, sep='\t')

# Display the first few rows
df.head()